In [29]:
# Khởi tạo 
from pathlib import Path

import numpy as np
import pandas as pd
from colorama import Fore, Style, init

pd.set_option("display.max_columns", None)


In [30]:
# Đọc dữ liệu gốc
RAW_PATH = Path("../../Data/Raw/uber.csv")
CLEAN_PATH = Path("../../Data/Clean/uber_cleaned.csv")

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file dữ liệu\n"
    )

raw_df = pd.read_csv(RAW_PATH)
df = raw_df.copy()
original_rows, original_columns = df.shape

print("Thông tin kiểu dữ liệu:")
df.info()
print(f"\nSố dòng trùng lặp: {df.duplicated().sum():,}")
print("\nTổng số ô có giá trị null theo cột:")
print(df.isnull().sum().to_string())


Thông tin kiểu dữ liệu:
<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 21 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Date                               150000 non-null  str    
 1   Time                               150000 non-null  str    
 2   Booking ID                         150000 non-null  str    
 3   Booking Status                     150000 non-null  str    
 4   Customer ID                        150000 non-null  str    
 5   Vehicle Type                       150000 non-null  str    
 6   Pickup Location                    150000 non-null  str    
 7   Drop Location                      150000 non-null  str    
 8   Avg VTAT                           139500 non-null  float64
 9   Avg CTAT                           102000 non-null  float64
 10  Cancelled Rides by Customer        10500 non-null   float64
 11  Reason for cancelling by C

In [31]:
# Xử lý dữ liệu trùng lặp và khuyết thiếu

# Loại bỏ các bản ghi trùng lặp 
df = df.drop_duplicates().copy()

# Các thuộc tính bắt buộc phải có trong dữ liệu 
required_cols = [
    'Booking ID', 'Date', 'Time', 'Booking Status', 'Vehicle Type', 'Pickup Location', 'Drop Location', 'Booking Value', 'Ride Distance'
]

# Thế các dòng có null chữ thành NA
string_cols = df.select_dtypes(include=['object']).columns
df[string_cols] = df[string_cols].fillna('NA')

# Thế các dòng có null số thành 0
number_cols = df.select_dtypes(include=['number']).columns
df[number_cols] = df[number_cols].fillna(0)

# xóa các dữ liệu không có ở các cột cần thiết
df = df.dropna(subset=required_cols).copy()
print("Thông tin kiểu dữ liệu:")
df.info()
print(f"\nSố dòng trùng lặp: {df.duplicated().sum():,}")
print("\nTổng số ô có giá trị null theo cột:")
print(df.isnull().sum().to_string())




C:\Users\manht\AppData\Local\Temp\ipykernel_20128\1802625238.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_cols = df.select_dtypes(include=['object']).columns


Thông tin kiểu dữ liệu:
<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 21 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Date                               150000 non-null  str    
 1   Time                               150000 non-null  str    
 2   Booking ID                         150000 non-null  str    
 3   Booking Status                     150000 non-null  str    
 4   Customer ID                        150000 non-null  str    
 5   Vehicle Type                       150000 non-null  str    
 6   Pickup Location                    150000 non-null  str    
 7   Drop Location                      150000 non-null  str    
 8   Avg VTAT                           150000 non-null  float64
 9   Avg CTAT                           150000 non-null  float64
 10  Cancelled Rides by Customer        150000 non-null  float64
 11  Reason for cancelling by C

In [32]:
# Chuẩn hóa kiểu dữ liệu
number_cols = [
    "Booking Value", "Ride Distance", "Avg VTAT", "Avg CTAT", "Driver Ratings", "Customer Rating",
]

# Ghép ngày và giờ thành thời điểm đón khách
df["pickup_datetime"] = pd.to_datetime(
    df["Date"].astype(str) + " " + df["Time"].astype(str),
    errors="coerce",
)

# Ép các cột kiểu số
for cols in number_cols:
    if cols in df.columns:
        df[cols] = pd.to_numeric(df[cols], errors="coerce")

# Bỏ dòng có thời gian, giá trị chuyến hoặc quãng đường không hợp lệ
df = df.dropna(subset=["pickup_datetime", "Booking Value", "Ride Distance"]).copy()
df = df[(df["Booking Value"] > 0) & (df["Ride Distance"] >= 0)].copy()

In [33]:
# Tạo, đọc bảng lookup location
LOCATION_MAPPING_PATH = Path("../../Data/Clean/location_mapping.csv")

def normalize_location(value):
    return " ".join(str(value).strip().lower().replace("-", " ").replace("_", " ").split())

#file location_mapping.csv là file location của nó và city, district được điền bởi AI
location_mapping = pd.read_csv(LOCATION_MAPPING_PATH, dtype=str).fillna("")

location_mapping["Location Key"] = location_mapping["Location"].map(normalize_location)

print(f"Khởi tạo bảng mapping cho các location trong file csv thành công")

Khởi tạo bảng mapping cho các location trong file csv thành công


In [34]:
# Tách các thành phần thời gian từ thời điểm đón khách
df["hour"] = df["pickup_datetime"].dt.hour
df["day"] = df["pickup_datetime"].dt.day
df["month"] = df["pickup_datetime"].dt.month
df["year"] = df["pickup_datetime"].dt.year
df["quarter"] = np.ceil(df["pickup_datetime"].dt.month / 3.0).astype(int)
df["day_of_week"] = df["pickup_datetime"].dt.day_name()

# Merge lookup cho location qua location mapping theo thứ tự: city -> district -> location
for location_column, prefix in [("Pickup Location", "pickup"), ("Drop Location", "drop")]:
    location_keys = df[location_column].map(normalize_location)
    lookup = location_mapping[["Location Key", "City", "District"]].rename(columns={"City": f"{prefix}_city","District": f"{prefix}_district"})
    
    df[f"{prefix}_city"] = location_keys.map(lookup.set_index("Location Key")[f"{prefix}_city"]).replace("", "Unknown").fillna("Unknown")
    
    df[f"{prefix}_district"] = location_keys.map(lookup.set_index("Location Key")[f"{prefix}_district"]).replace("", "Unknown").fillna("Unknown")

print("Các location đã được mapping:")
print(df[["pickup_city", "pickup_district", "Pickup Location", "drop_city", "drop_district", "Drop Location"]].head())
print("\nSố location chưa được điền mapping:")
print((df[["pickup_city", "pickup_district", "drop_city", "drop_district"]] == "Unknown").sum())

Các location đã được mapping:
  pickup_city pickup_district      Pickup Location drop_city drop_district  \
1       Delhi   Central Delhi        Shastri Nagar   Gurgaon      Gurugram   
2     Gurgaon        Gurugram              Khandsa     Delhi   South Delhi   
3       Delhi       New Delhi  Central Secretariat     Delhi         Delhi   
4       Delhi           Delhi     Ghitorni Village     Delhi     New Delhi   
5       Delhi     South Delhi                AIIMS   Gurgaon      Gurugram   

       Drop Location  
1  Gurgaon Sector 56  
2      Malviya Nagar  
3           Inderlok  
4        Khan Market  
5        Narsinghpur  

Số location chưa được điền mapping:
pickup_city        0
pickup_district    0
drop_city          0
drop_district      0
dtype: int64


In [37]:
# Xuất dữ liệu sạch và so sánh trước/sau
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)

comparison = pd.DataFrame(
    {
        " ": ["Rows", "Columns"],
        "Trước dọn dẹp": [original_rows, original_columns],
        "Sau dọn dẹp": [len(df), len(df.columns)],
    }
)

print(f"Đã lưu file tại: {CLEAN_PATH.resolve()}")
print("BẢNG SO SÁNH TRƯỚC VÀ SAU DỌN DẸP")
print(comparison.to_string(index=False))

for number, column in enumerate(df.columns, start=1):
    print(f"{number}. {column}")


Đã lưu file tại: D:\Storage\Documents\QLMH_2026-2027\OLAP\Đồ án real\Data\Clean\uber_cleaned.csv
BẢNG SO SÁNH TRƯỚC VÀ SAU DỌN DẸP
         Trước dọn dẹp  Sau dọn dẹp
   Rows         150000       102000
Columns             21           32
1. Date
2. Time
3. Booking ID
4. Booking Status
5. Customer ID
6. Vehicle Type
7. Pickup Location
8. Drop Location
9. Avg VTAT
10. Avg CTAT
11. Cancelled Rides by Customer
12. Reason for cancelling by Customer
13. Cancelled Rides by Driver
14. Driver Cancellation Reason
15. Incomplete Rides
16. Incomplete Rides Reason
17. Booking Value
18. Ride Distance
19. Driver Ratings
20. Customer Rating
21. Payment Method
22. pickup_datetime
23. hour
24. day
25. month
26. year
27. quarter
28. day_of_week
29. pickup_city
30. pickup_district
31. drop_city
32. drop_district
